# Atelier Préparation de Données Textuelles

Pipeline complet de préparation d'un corpus d'avis clients pour la classification de sentiment.

## Imports

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

## Partie 1 – Exploration du corpus

### 1.1 Charger les données CSV

In [3]:
df = pd.read_csv("../data/smart_reviews_raw.csv")
df.head()

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE.,positif,5,fr


### 1.2 Combien d'avis contient le dataset ?

In [6]:
print(f"Nombre d'avis : {len(df)}")

Nombre d'avis : 1200


### 1.3 Combien de colonnes possède-t-il ?

In [10]:
print(f"Nombre de colonnes : {df.shape[1]}")
print(df.columns.tolist())

Nombre de colonnes : 8
['id_avis', 'date', 'source', 'produit', 'texte', 'sentiment', 'note', 'langue']


### 1.4 Quel est le type de chaque colonne ?

In [14]:
df.dtypes

id_avis        str
date           str
source         str
produit        str
texte          str
sentiment      str
note         int64
langue         str
dtype: object

### 1.5 Existe-t-il des valeurs manquantes ?

In [16]:
manquantes = pd.DataFrame({
    "nb_manquantes": df.isna().sum(),
    "pourcentage": (df.isna().mean() * 100).round(2)
})
manquantes

,nb_manquantes,pourcentage
id_avis,0,0.00
date,0,0.00
source,0,0.00
produit,0,0.00
texte,5,0.42
sentiment,0,0.00
note,0,0.00
langue,0,0.00


### 1.6 Identifier quelques types de texte

On définit un masque booléen pour chaque type de texte à repérer : texte normal, vide, avec URL, mention, hashtag, emoji, beaucoup de ponctuation, en majuscules, avec répétition de caractères. On affiche ensuite quelques exemples de chaque type.

#### Mini-cours : les expressions régulières (regex)

Une **regex** est un motif qui décrit une forme de texte à repérer (une URL, une mention, un emoji…). Avec pandas, on l'utilise via `str.contains(motif, regex=True)`, qui renvoie `True` si le motif est trouvé dans le texte.

| Symbole | Signification | Exemple |
|---|---|---|
| `.` | n'importe quel caractère | `a.c` → "abc", "a9c" |
| `\w` | lettre, chiffre ou `_` | `\w` → "a", "7" |
| `\S` | tout caractère sauf l'espace | `\S` → "h", "/" |
| `[abc]` | un caractère parmi ceux listés | `[!?.]` → "!" ou "?" ou "." |
| `+` | 1 fois ou plus | `\w+` → "bonjour" |
| `{2,}` | 2 fois ou plus | `[!?]{2,}` → "!!!" |
| `\|` | ou | `http\|www` |
| `( )` | groupe (mémorisé) | `(.)` |
| `\1` | répète le groupe n°1 | `(.)\1` → "oo", "ss" |

**Les motifs utilisés ici**

- `@\w+` : un `@` suivi de lettres/chiffres, donc une **mention** (`@support`).
- `#\w+` : un `#` suivi de lettres/chiffres, donc un **hashtag** (`#top`).
- `https?://\S+` : "http" + "s" facultatif (`?` = 0 ou 1 fois) + `://` + tout jusqu'au prochain espace, donc une **URL**.
- `[!?.,;:]{2,}` : au moins 2 signes de ponctuation à la suite, donc de la **ponctuation répétée** (`!!!`, `???`).
- `(.)\1{2,}` : un caractère suivi de lui-même au moins 2 fois de plus, donc une **répétition de caractères** (`toooop`).
- `[\U0001F300-\U0001FAFF...]` : des **plages Unicode** où se trouvent les emojis.

In [19]:
# Version sans NaN pour appliquer les regex (df n'est pas modifié)
txt = df["texte"].fillna("").astype(str)

# Motifs regex
pat_url      = r"(https?://\S+|www\.\S+)"
pat_mention  = r"@\w+"
pat_hashtag  = r"#\w+"
pat_emoji    = (r"[\U0001F300-\U0001FAFF\U00002600-\U000027BF"
                r"\U0001F000-\U0001F2FF\U0001F900-\U0001F9FF]")
pat_ponct    = r"[!?.,;:]{2,}"       # ponctuation répétée (!!!, ???, ...)
pat_repet    = r"(.)\1{2,}"          # même caractère 3 fois ou plus (topppp, sooo)

# Masques par type de texte
masques = {
    "vide":                  df["texte"].isna() | (txt.str.strip() == ""),
    "avec URL":              txt.str.contains(pat_url, regex=True, case=False),
    "avec mention":          txt.str.contains(pat_mention, regex=True),
    "avec hashtag":          txt.str.contains(pat_hashtag, regex=True),
    "avec emojis":           txt.str.contains(pat_emoji, regex=True),
    "beaucoup de ponctuation": txt.str.contains(pat_ponct, regex=True),
    "en majuscules":         txt.str.isupper() & (txt.str.len() > 3),
    "répétition de caractères": txt.str.contains(pat_repet, regex=True),
}

# Texte "normal" = aucun des cas particuliers ci-dessus
masque_special = pd.concat(masques.values(), axis=1).any(axis=1)
masques = {"normal": ~masque_special, **masques}

# Effectifs par type
resume_types = pd.DataFrame({
    "nb_avis": {k: int(m.sum()) for k, m in masques.items()}
})
resume_types["pourcentage"] = (resume_types["nb_avis"] / len(df) * 100).round(2)
resume_types

C:\Users\other\AppData\Local\Temp\ipykernel_20796\877018334.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "avec URL":              txt.str.contains(pat_url, regex=True, case=False),
C:\Users\other\AppData\Local\Temp\ipykernel_20796\877018334.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "répétition de caractères": txt.str.contains(pat_repet, regex=True),


,nb_avis,pourcentage
normal,344,28.67
vide,6,0.50
avec URL,143,11.92
avec mention,128,10.67
avec hashtag,127,10.58
avec emojis,192,16.00
beaucoup de ponctuation,141,11.75
en majuscules,144,12.00
répétition de caractères,165,13.75


In [18]:
for nom, masque in masques.items():
    print(f"\n===== {nom.upper()} ({masque.sum()} avis) =====")
    exemples = df.loc[masque, "texte"].head(3)
    if exemples.empty:
        print("Aucun exemple")
    for e in exemples:
        print(" -", repr(e))


===== NORMAL (344 avis) =====
 - 'Très  bonne  expérience,  simple  et  efficace.'
 - 'Produit  parfait,  rien  à  signaler.'
 - 'Le  produit  correspond  globalement  à  la  description.'

===== VIDE (6 avis) =====
 - nan
 - nan
 - nan

===== AVEC URL (143 avis) =====
 - 'Produit parfait, rien à signaler. https://example.com/commande/17'
 - 'Très bonne expérience, simple et efficace. https://example.com/commande/31'
 - 'Je viens de recevoir le produit, à voir dans le temps. https://example.com/commande/32'

===== AVEC MENTION (128 avis) =====
 - '@client Livraison rapide et produit conforme à mes attentes.'
 - '@client Service client réactif et commande reçue rapidement.'
 - '@client La qualité est correcte sans être exceptionnelle.'

===== AVEC HASHTAG (127 avis) =====
 - 'Très satisfait de mon achat 👍 #avis'
 - "La batterie tient vraiment bien et l'écran est superbe. #avis"
 - 'Produit correct pour son prix. #avis'

===== AVEC EMOJIS (192 avis) =====
 - 'Très satisfait de mon achat